<div align="center">
  <img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Travel%20and%20places/Rocket.png" width="80" />
  <h1 style="color: #1D9E75; font-size: 3em;">📊 Customer Churn Prediction</h1>
  <p><strong>Telecom Industry Analysis Using Machine Learning</strong></p>
  <img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&height=100&section=header" width="100%" />
</div>

---

## 🛠️ Step 1: Import Libraries & Load Data

> "Good analysis starts with good tools!" - *Data Science Pro*

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/Hammer%20and%20Wrench.png" width="25" height="25" /> Setting up our workspace...

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
print("✅ All libraries loaded successfully!")

In [ ]:
df = pd.read_csv("churn.csv")

print("📂 Shape of Dataset:", df.shape)
print("\n🔍 Column names:\n", df.columns.tolist())
print("\n✨ First 5 rows:")
df.head()

In [ ]:
print("📄 Dataset Info:")
print(df.info())
print("\n❓ Missing values:\n", df.isnull().sum())
print("\n⚖️ Churn distribution:\n", df['Churn'].value_counts())
print("\n📉 Churn %:\n", df['Churn'].value_counts(normalize=True) * 100)

## 🎨 Step 2: Exploratory Data Analysis (EDA)

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/Magnifying%20Glass%20Tilted%20Right.png" width="25" height="25" /> Let's dive deep into the data patterns!

In [ ]:
plt.figure(figsize=(6,5))
df['Churn'].value_counts().plot(kind='bar', color=['#1D9E75','#E85D24'], edgecolor='none')
plt.title("Churn Distribution", fontsize=15)
plt.xlabel("Churn")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.boxplot(data=df, x='Churn', y=col, ax=ax, 
                palette={'No':'#1D9E75','Yes':'#E85D24'})
    ax.set_title(f'Churn vs {col}')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9,5))
ct = df.groupby(['Contract','Churn']).size().unstack()
ct.plot(kind='bar', color=['#1D9E75','#E85D24'], edgecolor='none', ax=plt.gca())
plt.title("Churn by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.legend(title="Churn")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='InternetService', hue='Churn', 
              palette={'No':'#1D9E75','Yes':'#E85D24'})
plt.title("Churn by Internet Service Type")
plt.tight_layout()
plt.show()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
num_df = df[['tenure','MonthlyCharges','TotalCharges']].copy()
num_df['Churn_num'] = (df['Churn'] == 'Yes').astype(int)
plt.figure(figsize=(7,5))
sns.heatmap(num_df.corr(), annot=True, fmt=".2f", cmap="YlGn")
plt.title("Correlation Heatmap")
plt.show()

## 🧹 Step 3: Data Preprocessing

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/Soap.png" width="25" height="25" /> Cleaning and encoding our data for machine learning.

In [ ]:
df.drop('customerID', axis=1, inplace=True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
print("✅ Shape after cleaning:", df.shape)

In [ ]:
binary_cols = ['Partner','Dependents','PhoneService','PaperlessBilling','Churn']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})
print("✨ Binary encoding complete.")

In [ ]:
cat_cols = ['MultipleLines','InternetService','OnlineSecurity', 
            'OnlineBackup','DeviceProtection','TechSupport', 
            'StreamingTV','StreamingMovies','Contract','PaymentMethod']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print("🚀 One-hot encoding complete!")
print("Final shape:", df.shape)

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']
print("Features:", X.shape)
print("Target distribution:\n", y.value_counts())

## ⚖️ Step 4: Handling Class Imbalance (SMOTE)

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/Triangular%20Ruler.png" width="25" height="25" /> Balancing the target classes using Synthetic Minority Over-sampling Technique.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train_bal).value_counts())

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal)
X_test_sc  = scaler.transform(X_test)
print("📏 Features scaled.")

## 🏗️ Step 5: Model Training & Comparison

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/Hammer.png" width="25" height="25" /> Training 3 powerful classifiers!

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest"      : RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost"            : XGBClassifier(use_label_encoder=False, 
                                         eval_metric='logloss', random_state=42)
}
results = {}
for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_sc, y_train_bal)
        preds = model.predict(X_test_sc)
        proba = model.predict_proba(X_test_sc)[:,1]
    else:
        model.fit(X_train_bal, y_train_bal)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:,1]
    
    auc = roc_auc_score(y_test, proba)
    results[name] = {"model": model, "preds": preds, "proba": proba, "auc": auc}
    print(f"\n✅ {name} trained. AUC: {auc:.4f}")

## 🏆 Step 6: Final Evaluation

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Activities/Trophy.png" width="25" height="25" /> Comparing the results!

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['preds'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Churn','Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Greens')
    ax.set_title(f"{name}\nAUC: {res['auc']:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
colors = ['#378ADD', '#1D9E75', '#E85D24']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", color=color, lw=2)
plt.plot([0,1],[0,1],'k--', lw=1, label='Random classifier')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(loc='lower right')
plt.show()

## 🌟 Step 7: Feature Importance

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Smilies/Star-Struck.png" width="25" height="25" /> What drives customer churn?

In [ ]:
rf_model = results['Random Forest']['model']
feat_imp = pd.DataFrame({
    'Feature'   : X.columns, 
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='YlGn_r')
plt.title("Top 15 Features Driving Customer Churn")
plt.xlabel("Importance Score")
plt.show()

<div align="center">
  <h3 style="color: #1D9E75;">🚀 Project Complete!</h3>
  <p>Ready to push to GitHub and impress recruiters.</p>
  <img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&height=100&section=footer" width="100%" />
</div>